# Citadel Colab TPU launcher (thin — first execution surface)
All logic lives in `citadel_tpu/`. Flow: fresh `citadel` + pinned Cymek runtime → handover → deps → preflight gate → probe → T0 → STOP unless T0 passes → export receipts. No secrets are used or printed. Cymek is never merged; its SHA is recorded per receipt.

In [ ]:
# 0. Fresh Citadel checkout at origin/citadel + pinned read-only Cymek runtime (public clone, no credentials)
import os, subprocess
repo = '/content/An-Ra-colab'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git','clone','--depth','50','-b','citadel','https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git',repo], check=True)
else:
    subprocess.run(['git','-C',repo,'fetch','origin','citadel','--depth','50'], check=True)
    subprocess.run(['git','-C',repo,'checkout','citadel'], check=True)
    subprocess.run(['git','-C',repo,'reset','--hard','origin/citadel'], check=True)
%cd /content/An-Ra-colab
os.environ['CITADEL_ROOT'] = '/content/An-Ra-colab'
import sys; sys.path.insert(0, '/content/An-Ra-colab')
from citadel_tpu import runtime_bootstrap as rb
rt_root, rt_sha = rb.ensure_cymek_runtime()
print('CITADEL_SHA=' + str(rb.citadel_sha()))
print('CYMEK_RUNTIME_SHA=' + str(rt_sha))
print('CYMEK_RUNTIME_PATH=' + str(rt_root))

In [ ]:
# 1. Inspect the handover (authoritative next-action source)
!sed -n '1,100p' agent.md

In [ ]:
# 2. Select PJRT TPU before importing torch-xla; inspect versions first (do not reinstall a working pair)
import os
os.environ.setdefault('PJRT_DEVICE', 'TPU')
os.environ['CITADEL_PLATFORM'] = 'colab'
print('PJRT_DEVICE', os.environ.get('PJRT_DEVICE'))
!python -c "import torch; print('torch', torch.__version__)" 2>&1 | tail -1
!python -c "import torch_xla; print('torch-xla', getattr(torch_xla, '__version__', 'unknown'))" 2>&1 | tail -1
!python -c "import numpy; print('numpy', numpy.__version__)" 2>&1 | tail -1

In [ ]:
# 3. Minimal conditional install (run ONLY if cell 2 showed a missing package)
# Do not reinstall a working torch/torch-xla pair.
# !pip install -q torch torch-xla numpy 2>&1 | tail -2

In [ ]:
# 4. Preflight gate: every import, file, API and the TPU itself verified. If this fails, do NOT run T0.
# (If cell 2 showed torch missing, run cell 3 first, then re-run this cell.)
import subprocess
r = subprocess.run(['python','-m','citadel_tpu.preflight'])
assert r.returncode == 0, 'PREFLIGHT failed — READY_FOR_T0=NO. Diagnose, do not escalate.'
print('PREFLIGHT READY_FOR_T0=YES')

In [ ]:
# 5. M0: environment receipt (fail-closed; ABORT_NO_TPU on CPU fallback)
from citadel_tpu import environment as env_mod
env = env_mod.main(out='docs/citadel/tpu_receipts/TPU_ENVIRONMENT.json', require_tpu=True, platform_override='colab')
print({k: env[k] for k in ('platform','accelerator_detected','xla_device_count','torch_version','torch_xla_version','probe_pass')})

In [ ]:
# 6. T0: single-device one-update certification (MINI_SPEC from pinned runtime, bucket 512, CE, one update)
from citadel_tpu import one_update
r0 = one_update.run(out='docs/citadel/tpu_receipts/TPU_ONE_UPDATE.json')
print({k: r0[k] for k in ('citadel_sha','cymek_runtime_sha','certification','loss','tokens_per_second','reload_identical')})

In [ ]:
# 7. STOP unless T0 passed. Export exact receipt files for operator transfer.
assert r0.get('certification') == 'PASS', 'T0 did not pass — STOP. Diagnose, do not escalate.'
from google.colab import files
files.download('docs/citadel/tpu_receipts/TPU_ENVIRONMENT.json')
files.download('docs/citadel/tpu_receipts/TPU_ONE_UPDATE.json')
print('exported; transfer these exact files back to the operator')

# T1 — Calculator canary (preregistered: experiments/T1/PLAN.md + AMENDMENT_001.md)
Primary metric is real held-out GENERATION exact-match (never teacher-forced CE). TEST is observed exactly twice total (untrained baseline + trained final) and never drives escalation. Do not edit thresholds; they are frozen in AMENDMENT_001.

In [ ]:
# A. T1 preflight gate: T0 receipt, runtime, generator, evaluator, TPU. If NO, do NOT train.
import subprocess
p1 = subprocess.run(['python','-m','citadel_tpu.calculator_preflight'])
assert p1.returncode == 0, 'T1 PREFLIGHT failed — READY_FOR_T1=NO. Diagnose, do not train.'
print('READY_FOR_T1=YES')

In [ ]:
# B. Data receipt preview (deterministic; overlap-guarded; no training)
from citadel_tpu import calculator_eval as cev
drc = cev.build_data_receipt()
print(drc['counts'], drc['overlap'], drc['scored_slice_counts'])

In [ ]:
# C. Full T1 run: untrained baseline → dev-gated ladder [5,20,100,200] → TEST-once → reload gate → receipt
from citadel_tpu import calculator_train
r1 = calculator_train.train(out='docs/citadel/tpu_receipts/TPU_CALCULATOR_CHECKPOINT.json')
print(r1['status'], r1['gate_rules'])

In [ ]:
# D. Receipt summary (read the file; no recompute, no threshold edits)
import json
rc = json.load(open('docs/citadel/tpu_receipts/TPU_CALCULATOR_CHECKPOINT.json'))
print('status:', rc['status'])
print('untrained:', rc['eval']['untrained_test'])
print('trained:', rc['eval']['trained_test'])
print('strongest null:', rc['strongest_heuristic_null'])
print('endpoint updates:', rc['training']['endpoint_updates'])

In [ ]:
# E. Reload verification display (prediction-hash identity, not just metric equality)
print('pre :', rc['pre_reload_prediction_sha256'])
print('post:', rc['post_reload_prediction_sha256'])
print('reload_identical:', rc['reload_identical'])

In [ ]:
# F. Export T1 receipt for operator transfer (checkpoint binary stays out of git)
files.download('docs/citadel/tpu_receipts/TPU_CALCULATOR_CHECKPOINT.json')
print('exported; transfer this exact file back to the operator')